In [1]:
from datetime import datetime
import glob
import os
from dotenv import load_dotenv
import pandas as pd

load_dotenv()


# Define input (Silver) and output (Bronze) directory paths


# silver_dir = os.getenv("SILVERALL")
bronze_dir = os.getenv("BRONZENEW")
silver_dir = os.getenv("SILVER")

# Update silver_dir to include a subfolder named with the current date (e.g., YYYY-MM-DD or YYYYMMDD)
current_date_folder = datetime.now().strftime("%Y-%m-%d")
bronze_dir = os.path.join(bronze_dir, current_date_folder)

# Create Bronze directory if it doesn't already exist
if not os.path.exists(silver_dir):
    os.makedirs(silver_dir)
    print(f"Created directory: {silver_dir}")

# Find all Excel and CSV files inside the Silver folder
file_patterns = [
    os.path.join(bronze_dir, "*.xlsx"),
    os.path.join(bronze_dir, "*.xls"),
    os.path.join(bronze_dir, "*.csv"),
]

all_files = []
for pattern in file_patterns:
    all_files.extend(glob.glob(pattern))

print(f"Found {len(all_files)} file(s) in Silver layer:")
for f in all_files:
    print(f" - {os.path.basename(f)}")

if not all_files:
    print("\n❌ No matching Excel or CSV files were found in the Silver folder.")
else:
    dataframes = []

    # Read each file and append to our list
    for file_path in all_files:
        try:
            filename = os.path.basename(file_path)
            if file_path.endswith(".csv"):
                df = pd.read_csv(file_path)
            else:
                df = pd.read_excel(file_path)

            # --- Drop existing S. No. columns if present ---
            sno_variations = [
                "S. No.",
                "S.No.",
                "S.No",
                "S. No",
                "SNo",
                "S_No",
                "s.no.",
                "s.no",
            ]
            cols_to_drop = [
                col for col in df.columns if str(col).strip() in sno_variations
            ]
            if cols_to_drop:
                df.drop(columns=cols_to_drop, inplace=True)

            # Optionally add a metadata column tracking the source filename
            df["Source_File"] = filename

            dataframes.append(df)
            print(f"✓ Successfully loaded: {filename} ({len(df)} rows)")
        except Exception as e:
            print(f"❌ Error reading {os.path.basename(file_path)}: {e}")

    # Merge/Concatenate all DataFrames into one
    if dataframes:
        merged_df = pd.concat(dataframes, ignore_index=True)

        # --- Generate new continuous S. No. column ---
        merged_df.insert(0, "S. No.", range(1, len(merged_df) + 1))

        # Define output destination file path
        timestamp = datetime.now().strftime("%Y%m%d")
        output_file_path = os.path.join(silver_dir, f"{timestamp}.xlsx")

        # Save merged dataframe to Excel
        merged_df.to_excel(output_file_path, index=False)
        print(f"\n✅ Merge complete! Total combined rows: {len(merged_df)}")
        print(f"📁 Output file created at:\n   {output_file_path}")
    else:
        print("\n❌ Failed to parse any data from the identified files.")

Found 31 file(s) in Silver layer:
 - MOEFCC_20260806_183602.xlsx
 - SEIAA_ANDAMAN_AND_NICOBAR_ISLANDS_20260806_181145.xlsx
 - SEIAA_ANDHRA_PRADESH_20260806_175217.xlsx
 - SEIAA_ARUNACHAL_PRADESH_20260806_181023.xlsx
 - SEIAA_ASSAM_20260806_185152.xlsx
 - SEIAA_BIHAR_20260806_183009.xlsx
 - SEIAA_CHHATTISGARH_20260806_174540.xlsx
 - SEIAA_DELHI_20260806_181430.xlsx
 - SEIAA_GOA_20260806_180349.xlsx
 - SEIAA_GUJARAT_20260806_175547.xlsx
 - SEIAA_HARYANA_20260806_180003.xlsx
 - SEIAA_HIMACHAL_PRADESH_20260806_174938.xlsx
 - SEIAA_JAMMU_AND_KASHMIR_20260806_175309.xlsx
 - SEIAA_JHARKHAND_20260806_184442.xlsx
 - SEIAA_KARNATAKA_20260806_175404.xlsx
 - SEIAA_KERALA_20260806_190512.xlsx
 - SEIAA_MADHYA_PRADESH_20260806_175708.xlsx
 - SEIAA_MAHARASHTRA_20260806_175713.xlsx
 - SEIAA_MANIPUR_20260806_181513.xlsx
 - SEIAA_MEGHALAYA_20260806_175525.xlsx
 - SEIAA_ODISHA_20260806_174522.xlsx
 - SEIAA_PUDUCHERRY_20260806_180626.xlsx
 - SEIAA_PUNJAB_20260806_174414.xlsx
 - SEIAA_RAJASTHAN_20260806_181